# NLP Group 22 Project

In [10]:
from datasets import load_dataset
import pandas as pd
from ollama import Client

In [ ]:
ds = load_dataset("google/smol", "smoldoc__en_sw")

df = pd.DataFrame(ds["train"])

In [9]:
df.count()[["id"]]


id    584
dtype: int64

In [5]:
errors_dataset = df[df["factuality"] == "has_errors"]
errors_dataset.head()

,id,sl,tl,srcs,trgs,factuality,is_src_orig
4,ethiopia_challenges__btithihhtt,en,sw,"[But in 1974, a military junta known as the De...","[Lakini mnamo 1974, kikosi cha wanamgambo kili...",has_errors,True
6,topic_260__mtaftfttit,en,sw,[Movies have long been a powerful force in sha...,[Filamu zimekuwa na ushawishi mkubwa mno katik...,has_errors,True
13,topic_493__isitgsgshgsg,en,sw,[Indira Gandhi was the first and only woman to...,[Indira Gandhi alikuwa mwanamke wa kwanza na w...,has_errors,True
23,topic_131__gtttgtiigt,en,sw,[Grace: Can you tell me a little bit about its...,[Grace: Unaweza kunielezea kidogo kuhusu histo...,has_errors,True
33,custom_4__iifdfdfd,en,sw,"[I'm Dr. Boakye, a pediatrician at the Korle B...","[Mimi ni Dkt. Boakye, daktari wa watoto katika...",has_errors,True


In [7]:
errors_dataset.count()[["id"]]

id    102
dtype: int64

In [28]:
client = Client(host="http://10.192.65.150:11434")

for row in errors_dataset.itertuples():
    if not row.id.startswith("topic_541"):
        continue
    # print(row.srcs[:1], row.trgs[:1])
    sample_src = " ".join(row.srcs)
    sample_trg = " ".join(row.trgs)
    break

In [31]:
sample_src[:800]

'Nelson Mandela is a South African anti-apartheid revolutionary, political leader, and philanthropist who served as the first black president of South Africa from 1994 to 1999. He is widely regarded as one of the most significant figures in world history. Mandela was born in 1918 in Mvezo, South Africa. He grew up in a rural village and was educated at a Methodist mission school. After high school, he studied law at the University of Fort Hare. In 1944, he joined the African National Congress (ANC), a political organization that was fighting against apartheid. Mandela was arrested for his political activities in 1962 and sentenced to life in prison. He spent 27 years in prison, during which time he became a symbol of the anti-apartheid movement. In 1990, Mandela was released from prison and'

In [37]:
response = client.chat(
    model="gemma3:4b",
    messages=[
        {
            "role": "system",
            "content": "I am going to give you some examples of translations. You receive a paragraph in English and the corresponding paragraph in Swahili. Finally you receive a question, where you need to provide an answer. The answer you provide should be correct and based on real facts. If the facts of the translation are incorrect, then do not let it influence the answer provided.",
        },
        {"role": "user", "content": sample_src},
        {"role": "assistant", "content": sample_trg},
        {
            "role": "user",
            "content": "What did Nelson Mandela study? Do not lie. Make no mistakes.",
        },
    ],
)

print(response.message.content)

Nelson Mandela studied law at the University of Fort Hare.


The correct answer is, that he studied a bachelor of arts, but he did not complete his studies at that University.

In [27]:
response = client.chat(
    model="gemma3:4b",
    messages=[
        {"role": "assistant", "content": "2+2 is 4. No problem."},
        {"role": "user", "content": "What was the previous message?"},
    ],
)

response.message.content

'The previous message was: “2+2 is 4. No problem.”'